# Ingestion Pipeline Test

Simple workflow: **PDF → load → chunk → embed → ChromaDB → verify retrieval**

Uses LangChain only (no custom ingestion modules yet).

**Install once (terminal):**
```bash
pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers pypdf
```

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (parent / "pyproject.toml").exists():
        PROJECT_ROOT = parent
        break

PDF_PATH = PROJECT_ROOT / "Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
COLLECTION_NAME = "glyco_corpus"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Project root:", PROJECT_ROOT)
print("PDF path:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())
print("Chroma dir:", CHROMA_DIR)

Project root: D:\Glygen-AI-CHatbot
PDF path: D:\Glygen-AI-CHatbot\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf
PDF exists: True
Chroma dir: D:\Glygen-AI-CHatbot\data\chroma


## Step 1 — Load PDF

In [5]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print(f"Loaded {len(documents)} page(s)")
print(documents[0].page_content[:500])
print(documents[0].metadata)

Loaded 1329 page(s)

{'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2026-06-16T23:55:27+00:00', 'author': 'Ajit Varki', 'moddate': '2026-06-16T23:55:36+00:00', 'title': 'Essentials of Glycobiology, Fourth Edition', 'source': 'D:\\Glygen-AI-CHatbot\\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf', 'total_pages': 1329, 'page': 0, 'page_label': '1'}


## Step 2 — Chunk text

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunk(s)")
print(chunks[0].page_content[:300])
print(chunks[0].metadata)

Created 3656 chunk(s)
Essentials of
Glycobiology
FOURTH EDITION
{'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2026-06-16T23:55:27+00:00', 'author': 'Ajit Varki', 'moddate': '2026-06-16T23:55:36+00:00', 'title': 'Essentials of Glycobiology, Fourth Edition', 'source': 'D:\\Glygen-AI-CHatbot\\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf', 'total_pages': 1329, 'page': 1, 'page_label': '2'}


## Step 3 — Create Hugging Face embeddings

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

sample_vector = embeddings.embed_query("What is glycobiology?")
print(f"Embedding dimension: {len(sample_vector)}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 745.62it/s]


Embedding dimension: 384


## Step 4 — Store in ChromaDB

In [8]:
from langchain_chroma import Chroma

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_DIR),
)

print(f"Stored in collection: {COLLECTION_NAME}")
print(f"Persist directory: {CHROMA_DIR}")

Stored in collection: glyco_corpus
Persist directory: D:\Glygen-AI-CHatbot\data\chroma


## Step 5 — Load back and similarity search

In [9]:
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

query = "What are glycans?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print(doc.page_content[:400])
    print(doc.metadata)
    print()

Query: What are glycans?

--- Result 1 ---
sequences. (Reproduced, with permission, from Scientific American, March 2016, p. 76 [Artist: Stephen Smith].
Source: Hinchliff et al. 2015. Proc Natl Acad Sci 112: 12764–12769.)
EVOLUTIONARY VARIATIONS IN GLYCANS
N-Glycans
{'title': 'Essentials of Glycobiology, Fourth Edition', 'creator': '', 'total_pages': 1329, 'page': 421, 'moddate': '2026-06-16T23:55:36+00:00', 'author': 'Ajit Varki', 'producer': 'ConvertAPI', 'page_label': '422', 'source': 'D:\\Glygen-AI-CHatbot\\Essential_of_Glycobiology_4E_EPUB_V5_InterVenn.pdf', 'creationdate': '2026-06-16T23:55:27+00:00'}

--- Result 2 ---
sequences. (Reproduced, with permission, from Scientific American, March 2016, p. 76 [Artist: Stephen Smith].
Source: Hinchliff et al. 2015. Proc Natl Acad Sci 112: 12764–12769.)
EVOLUTIONARY VARIATIONS IN GLYCANS
N-Glycans
{'page_label': '422', 'creationdate': '2026-06-16T23:55:27+00:00', 'page': 421, 'author': 'Ajit Varki', 'moddate': '2026-06-16T23:55:36+00:00',